# Chapter 2: FLAML — Lightweight, Cost-Aware AutoML

### Why This Matters

FLAML ("Fast and Lightweight AutoML," from Microsoft) is built for exactly the situation an individual analyst on a laptop, rather than a large compute cluster, actually faces: a fixed, small time or resource budget, after which you want the *best model achievable in that budget* rather than a search that runs indefinitely.

In [ ]:
from flaml import AutoML
from sklearn.model_selection import train_test_split

X = pd.get_dummies(df.drop(columns=["churn"]), drop_first=True)
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

automl = AutoML()
automl.fit(
    X_train, y_train,
    task="classification",
    metric="roc_auc",
    time_budget=60,                                    # a HARD wall-clock budget, in seconds
    estimator_list=["lgbm", "xgboost", "rf", "extra_tree"],
    verbose=0
)

print("Best model:", automl.best_estimator)
print("Best config:", automl.best_config)

`time_budget=60` is FLAML's defining feature — rather than specifying "try this many trials" (as Optuna's `n_trials` does), you specify a wall-clock time limit, and FLAML's internal cost-aware search allocates that budget intelligently across trying different model families and hyperparameter configurations, favoring cheaper models/configurations early and only spending more time on expensive ones once they've shown promise. This framing — "give me the best you can in exactly N minutes" — maps directly onto how a real analyst has to think about time budgets under deadline pressure, which is exactly the design philosophy behind the library.

💡 **Practical exercise worth actually running** (from the source material): try `time_budget=60` and then `time_budget=600` (10 minutes) on the same problem, and compare the ROC-AUC improvement. This "accuracy gained per additional minute of compute" curve typically shows steep early gains that flatten out — a genuinely useful thing to have seen with your own eyes once, so you have realistic intuition for how much a longer AutoML run is actually likely to help on your specific data going forward.

### Cheat Sheet — FLAML

| Task | Code |
|---|---|
| Initialize | `automl = AutoML()` |
| Run search | `automl.fit(X, y, task="classification", time_budget=60, metric="roc_auc")` |
| Best model name | `automl.best_estimator` |
| Best hyperparameters | `automl.best_config` |

---